# Protein Model Evaluation and Comparison

This notebook provides a step-by-step evaluation and comparison of two trained protein prediction models. The workflow includes loading models, preparing data, optimizing thresholds, evaluating performance on residue and domain levels, and analyzing confusion patterns between protein families.

**Outline:**
1. Import Required Libraries
2. Load Trained Models
3. Load and Prepare Datasets
4. Obtain Model Predictions and Probabilities (Softmax)
5. Find Optimal Threshold on Train & Validation Sets
6. Evaluate Per-Class Metrics (Accuracy, Precision, Recall, F1)
7. Test Set Evaluation: Predictions and Metrics
8. Measure Inference Runtime and GPU Usage
9. Calculate Model Size (Number of Parameters)
10. Compute Confusion Matrix for Residue-Level Predictions
11. Domain-Level Prediction Evaluation
12. Analyze Confusion Patterns Between Protein Families

## 1. Import Required Libraries
Import all necessary libraries for data handling, model loading, evaluation, and visualization.

In [ ]:
# Standard libraries
import os
import sys
import time
import tracemalloc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Sklearn metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Custom imports (adjust paths as needed)
sys.path.append(os.path.abspath('../models'))
from Dataset import ProteinCSVWindowDataset, collate_fn_window
from original import OriginalModel
from small import SmallModel

## 2. Load Trained Models
Load both final trained models from their .pth files and set them to evaluation mode.

In [ ]:
# Specify model checkpoint paths
MODEL1_PATH = '../checkpoints/original_final.pth'  # Update with actual path
MODEL2_PATH = '../checkpoints/small_final.pth'     # Update with actual path

# Instantiate models
model1 = OriginalModel(num_classes=2)
model2 = SmallModel(num_classes=2)

# Load weights
model1.load_state_dict(torch.load(MODEL1_PATH, map_location='cpu'))
model2.load_state_dict(torch.load(MODEL2_PATH, map_location='cpu'))

# Set to evaluation mode
model1.eval()
model2.eval()

## 3. Load and Prepare Datasets
Load train, validation, and test datasets from CSV files. Prepare DataLoader objects for each split.

In [ ]:
# File paths
TRAIN_CSV = '../train.csv'
VAL_CSV = '../val.csv'
TEST_CSV = '../test.csv'

# Create datasets
train_dataset = ProteinCSVWindowDataset(TRAIN_CSV)
val_dataset = ProteinCSVWindowDataset(VAL_CSV)
test_dataset = ProteinCSVWindowDataset(TEST_CSV)

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_window)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_window)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_window)

## 4. Obtain Model Predictions and Probabilities (Softmax)
Run inference on train and validation sets for both models. Apply softmax to outputs to get class probabilities.

In [ ]:
def get_probs_and_labels(model, loader, device='cpu'):
    model.to(device)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['embeddings'].to(device)
            labels = batch['labels'].to(device)
            logits, _ = model(inputs)
            probs = F.softmax(logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

# Combine train and val for threshold search
combined_loader = DataLoader(torch.utils.data.ConcatDataset([train_dataset, val_dataset]), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_window)

probs1, labels1 = get_probs_and_labels(model1, combined_loader)
probs2, labels2 = get_probs_and_labels(model2, combined_loader)

## 5. Find Optimal Threshold on Train & Validation Sets
Use grid search to find the optimal probability threshold for classification, maximizing F1-score.

In [ ]:
from sklearn.metrics import f1_score

def find_best_threshold(probs, labels, metric=f1_score):
    best_thr, best_score = 0.5, 0
    y_true = labels.argmax(axis=-1).flatten()
    for thr in np.linspace(0.1, 0.95, 18):
        y_pred = probs.argmax(axis=-1)
        mask = probs.max(axis=-1) >= thr
        y_pred_thr = np.where(mask, y_pred, -1)
        valid = y_pred_thr != -1
        if valid.sum() == 0:
            continue
        score = metric(y_true[valid], y_pred_thr[valid], average='macro')
        if score > best_score:
            best_score = score
            best_thr = thr
    return best_thr, best_score

best_thr1, best_f1_1 = find_best_threshold(probs1, labels1)
best_thr2, best_f1_2 = find_best_threshold(probs2, labels2)
print(f"Model1 best threshold: {best_thr1:.2f}, F1: {best_f1_1:.4f}")
print(f"Model2 best threshold: {best_thr2:.2f}, F1: {best_f1_2:.4f}")

## 6. Evaluate Per-Class Metrics (Accuracy, Precision, Recall, F1) on Test Set
Calculate per-class accuracy, precision, recall, and F1-score for both models on the test set using the optimal threshold found from train+val.

In [ ]:
probs1_test, labels1_test = get_probs_and_labels(model1, test_loader)
probs2_test, labels2_test = get_probs_and_labels(model2, test_loader)

In [ ]:
def per_class_metrics(probs, labels, threshold):
    y_true = labels.argmax(axis=-1).flatten()
    y_pred = probs.argmax(axis=-1)
    mask = probs.max(axis=-1) >= threshold
    y_pred_thr = np.where(mask, y_pred, -1)
    valid = y_pred_thr != -1
    report = classification_report(y_true[valid], y_pred_thr[valid], output_dict=True)
    return report

report1_test = per_class_metrics(probs1_test, labels1_test, best_thr1)
report2_test = per_class_metrics(probs2_test, labels2_test, best_thr2)
print("Model1 Test classification report:")
display(pd.DataFrame(report1_test).T)
print("Model2 Test classification report:")
display(pd.DataFrame(report2_test).T)

## 7. Test Set Evaluation: Predictions and Metrics
Run inference on the test set for both models. Compute accuracy, precision, recall, F1-score (macro), and confusion matrix at the residue level.

In [ ]:
def test_metrics(probs, labels, threshold):
    y_true = labels.argmax(axis=-1).flatten()
    y_pred = probs.argmax(axis=-1)
    mask = probs.max(axis=-1) >= threshold
    y_pred_thr = np.where(mask, y_pred, -1)
    valid = y_pred_thr != -1
    acc = accuracy_score(y_true[valid], y_pred_thr[valid])
    prec = precision_score(y_true[valid], y_pred_thr[valid], average='macro', zero_division=0)
    rec = recall_score(y_true[valid], y_pred_thr[valid], average='macro', zero_division=0)
    f1 = f1_score(y_true[valid], y_pred_thr[valid], average='macro', zero_division=0)
    cm = confusion_matrix(y_true[valid], y_pred_thr[valid])
    return acc, prec, rec, f1, cm

acc1, prec1, rec1, f1_1, cm1 = test_metrics(probs1_test, labels1_test, best_thr1)
acc2, prec2, rec2, f1_2, cm2 = test_metrics(probs2_test, labels2_test, best_thr2)
print(f"Model1 Test - Acc: {acc1:.4f}, Prec: {prec1:.4f}, Rec: {rec1:.4f}, F1: {f1_1:.4f}")
print(f"Model2 Test - Acc: {acc2:.4f}, Prec: {prec2:.4f}, Rec: {rec2:.4f}, F1: {f1_2:.4f}")

## 8. Measure Inference Runtime and GPU Usage
Measure inference time using time.perf_counter() and GPU memory usage with tracemalloc.get_traced_memory() for both models on the test set.

In [ ]:
def measure_inference_time_and_memory(model, loader, threshold, device='cpu'):
    model.to(device)
    tracemalloc.start()
    start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            inputs = batch['embeddings'].to(device)
            logits, _ = model(inputs)
            _ = F.softmax(logits, dim=-1)
    end = time.perf_counter()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return end - start, peak / 1e6  # seconds, MB

time1, mem1 = measure_inference_time_and_memory(model1, test_loader, best_thr1)
time2, mem2 = measure_inference_time_and_memory(model2, test_loader, best_thr2)
print(f"Model1 inference time: {time1:.2f}s, peak memory: {mem1:.2f}MB")
print(f"Model2 inference time: {time2:.2f}s, peak memory: {mem2:.2f}MB")

## 9. Calculate Model Size (Number of Parameters)
Count and display the number of trainable parameters for each model.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model1 (Original) parameters: {count_parameters(model1):,}")
print(f"Model2 (Small) parameters: {count_parameters(model2):,}")

## 10. Compute Confusion Matrix for Residue-Level Predictions
Plot and analyze confusion matrices for residue-level predictions for both models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm1, annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Model1 Confusion Matrix')
sns.heatmap(cm2, annot=True, fmt='d', ax=axes[1], cmap='Greens')
axes[1].set_title('Model2 Confusion Matrix')
plt.show()

## 11. Domain-Level Prediction Evaluation
Compare predicted domains with annotated domains in the test set and evaluate domain-level accuracy. This typically involves grouping residue-level predictions into domains and comparing them to annotated domain boundaries.

**Suggestions:**
- Define a function to extract domains from residue-level predictions (e.g., consecutive residues with the same label).
- Compare predicted domains to true domains using overlap, Jaccard index, or boundary accuracy.
- Compute domain-level precision, recall, and F1-score.

In [ ]:
# Example structure for domain-level evaluation

def extract_domains(labels):
    """Extract domain regions from a sequence of labels."""
    domains = []
    current_label = None
    start = 0
    for i, label in enumerate(labels):
        if label != current_label:
            if current_label is not None:
                domains.append((current_label, start, i-1))
            current_label = label
            start = i
    domains.append((current_label, start, len(labels)-1))
    return domains

# Example usage:
# y_true_seq = labels1_test.argmax(axis=-1)[0]  # for one protein
# y_pred_seq = probs1_test.argmax(axis=-1)[0]
# true_domains = extract_domains(y_true_seq)
# pred_domains = extract_domains(y_pred_seq)
# Now compare true_domains and pred_domains for overlap, etc.

print("Domain-level evaluation: Use extract_domains and compare predicted/true domains as needed.")

## 12. Analyze Confusion Patterns Between Protein Families
Compute and visualize confusion matrices for actual vs predicted protein families. Analyze common confusion patterns.

**Suggestions:**
- Ensure your dataset includes a family label for each protein.
- Aggregate predictions at the protein level (e.g., majority vote or max probability for each protein).
- Compute a confusion matrix using true and predicted family labels.
- Visualize the confusion matrix and highlight common confusions.

In [ ]:
# Example structure for family-level confusion matrix
# Assume you have lists: true_families, pred_families (one per protein)
from sklearn.metrics import confusion_matrix

# true_families = [...]  # List of true family labels for each protein
# pred_families = [...]  # List of predicted family labels for each protein
# family_names = [...]   # List of all family names (for axis labels)

# cm_family = confusion_matrix(true_families, pred_families, labels=family_names)
# plt.figure(figsize=(8,6))
# sns.heatmap(cm_family, annot=True, fmt='d', xticklabels=family_names, yticklabels=family_names, cmap='Purples')
# plt.xlabel('Predicted Family')
# plt.ylabel('True Family')
# plt.title('Family-level Confusion Matrix')
# plt.show()

print("Family-level confusion analysis: Fill true_families and pred_families, then use the above code to visualize and analyze confusion patterns.")